# 09 Model Versioning & Pipeline Persistence
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Persist the winning production pipeline, metadata manifest, and feature schema to `models/v1/` and `models/`.


In [2]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score

DATA_PROCESSED = "../data/processed"
MODELS_DIR = "../models"
V1_DIR = os.path.join(MODELS_DIR, "v1")
os.makedirs(V1_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(DATA_PROCESSED, "attrition_features_engineered.csv"))

SENSITIVE_ATTRS = ['gender', 'marital_status']
TARGET_COLS = ['attrition', 'attrition_binary']
ID_COLS = ['employee_id']

feature_cols = [c for c in df.columns if c not in SENSITIVE_ATTRS + TARGET_COLS + ID_COLS]
X = df[feature_cols]
y = df['attrition_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42))
])

final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
    "pr_auc": round(float(average_precision_score(y_test, y_prob)), 4),
    "recall": round(float(recall_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred)), 4)
}

metadata = {
    "model_name": "Enterprise HR Attrition Predictor",
    "version": "v1.0.0",
    "algorithm": "RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced')",
    "created_at": datetime.now().isoformat(),
    "training_samples": len(X_train),
    "test_samples": len(X_test),
    "feature_columns": feature_cols,
    "numerical_features": num_cols,
    "categorical_features": cat_cols,
    "metrics": metrics,
    "risk_thresholds": {
        "low": "< 0.35",
        "medium": "0.35 - 0.65",
        "high": ">= 0.65"
    }
}

# Save artifacts
pipeline_path_v1 = os.path.join(V1_DIR, "attrition_pipeline.joblib")
meta_path_v1 = os.path.join(V1_DIR, "metadata.json")
pipeline_path_root = os.path.join(MODELS_DIR, "attrition_pipeline.joblib")
meta_path_root = os.path.join(MODELS_DIR, "metadata.json")

joblib.dump(final_pipeline, pipeline_path_v1)
joblib.dump(final_pipeline, pipeline_path_root)

with open(meta_path_v1, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
with open(meta_path_root, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved production pipeline to: {pipeline_path_v1}")
print(f"✓ Saved metadata to: {meta_path_v1}")
print(f"Model Metrics: {metrics}")


✓ Saved production pipeline to: ../models\v1\attrition_pipeline.joblib
✓ Saved metadata to: ../models\v1\metadata.json
Model Metrics: {'roc_auc': 0.7853, 'pr_auc': 0.4612, 'recall': 0.2766, 'precision': 0.4643, 'f1_score': 0.3467}
